# HC3 — Dual Logistic Regression with Tunable Weight

## Idea

Instead of fusing TF-IDF and custom features into one model, we train two **separate**
logistic regression models:

- `model_tfidf` — trained on TF-IDF features only (learns vocabulary/phrase patterns)
- `model_custom` — trained on stylometric features only (learns writing style patterns)

At inference we combine their predicted probabilities with a tunable weight `alpha`:

```
final_prob = alpha * prob_custom + (1 - alpha) * prob_tfidf
```

- `alpha = 0.0` → pure TF-IDF (original behaviour, likely overfits to HC3 phrases)
- `alpha = 1.0` → pure custom features (stylometric only, more generalisable)
- `alpha = 0.5` → equal weight (default starting point)

We find the best `alpha` by scanning 0.0 → 1.0 in steps of 0.05 using the **validation
fold only** (never the test set). This lets you see clearly how much each side contributes
and choose a value that trades benchmark accuracy for real-world robustness.

---
## Step 0 — Imports

In [ ]:
import ast
import joblib
import numpy as np
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

---
## Step 1 — Load and clean data

In [ ]:
def clean_text(x):
    if isinstance(x, list):
        return ' '.join(map(str, x))
    if isinstance(x, dict):
        return ' '.join(map(str, x.values()))
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return ' '.join(map(str, parsed))
            return str(parsed)
        except (ValueError, SyntaxError):
            return x
    return str(x)


df = pd.read_csv('cleaned_data.csv')
df['answers'] = df['answers'].str.lower().apply(clean_text)

print(f'Shape: {df.shape}')
print(df['label'].value_counts())

---
## Step 2 — Feature extraction

Same sentence-split → per-sentence features → aggregate pipeline as your original.
Raw counts dropped; ratios only (no `spelling_error_ratio` — it was always 1.0).

In [ ]:
nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])
nlp.add_pipe('sentencizer')


def load_word_list(path):
    with open(path, 'r', encoding='utf-8') as f:
        return set(line.strip().lower() for line in f if line.strip())


chat_words        = load_word_list('chat_words.txt')
function_words    = load_word_list('function_words.txt')
discourse_markers = load_word_list('discourse_markers.txt')


def split_sentences(text):
    doc = nlp(text)
    return [s.text.strip() for s in doc.sents if s.text.strip()]


def extract_features(text):
    doc          = nlp(text)
    tokens       = [t.text.lower() for t in doc if not t.is_space]
    alpha_tokens = [t.text.lower() for t in doc if t.is_alpha]
    total_tokens = len(tokens)
    total_alpha  = len(alpha_tokens)

    if total_tokens == 0 or total_alpha == 0:
        return dict(chat_word_ratio=0., punct_ratio=0., ttr=0.,
                    function_word_ratio=0., discourse_ratio=0.,
                    sentence_length=0., avg_word_length=0.)

    return {
        'chat_word_ratio'    : sum(1 for t in tokens if t in chat_words)     / total_tokens,
        'punct_ratio'        : sum(1 for t in doc if t.is_punct)              / total_tokens,
        'ttr'                : len(set(alpha_tokens))                          / total_alpha,
        'function_word_ratio': sum(1 for t in tokens if t in function_words)  / total_tokens,
        'discourse_ratio'    : sum(1 for t in tokens if t in discourse_markers)/ total_tokens,
        'sentence_length'    : float(total_tokens),
        'avg_word_length'    : sum(len(t) for t in alpha_tokens)               / total_alpha,
    }


def aggregate_sentence_features(sentences):
    feats = [extract_features(s) for s in sentences]
    if not feats:
        return {}
    fd = pd.DataFrame(feats)
    return {
        'chat_word_ratio'    : fd['chat_word_ratio'].mean(),
        'punct_ratio'        : fd['punct_ratio'].mean(),
        'ttr'                : fd['ttr'].mean(),
        'function_word_ratio': fd['function_word_ratio'].mean(),
        'discourse_ratio'    : fd['discourse_ratio'].mean(),
        'sentence_length_mean': fd['sentence_length'].mean(),
        'sentence_length_std' : fd['sentence_length'].std(ddof=1) if len(fd) > 1 else 0.0,
        'avg_word_length'    : fd['avg_word_length'].mean(),
    }


def build_features(df_in):
    rows = []
    for _, row in df_in.iterrows():
        feats = aggregate_sentence_features(split_sentences(row['answers']))
        feats['label'] = row['label']
        rows.append(feats)
    return pd.DataFrame(rows)


print('Running feature extraction...')
df_feat = build_features(df)
print(f'Done. Shape: {df_feat.shape}')
df_feat.head()

---
## Step 3 — Three-way split: train / validation / test

We need three splits:

- **Train (64%)** — fit both models and the TF-IDF vectoriser
- **Validation (16%)** — find the best `alpha` weight (never used for model fitting)
- **Test (20%)** — final honest evaluation only, never touched until the end

Using a separate validation set for alpha search is critical. If we used the test set to
pick alpha we would be leaking test information into the model selection process.

In [ ]:
X_ans    = df['answers'].reset_index(drop=True)
y        = df['label'].reset_index(drop=True)
X_custom = df_feat.drop(columns=['label']).reset_index(drop=True)

# First split off the test set (20%)
X_ans_tmp, X_ans_test, X_cust_tmp, X_cust_test, y_tmp, y_test = train_test_split(
    X_ans, X_custom, y,
    test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Split remainder into train (80% of tmp = 64% total) and val (20% of tmp = 16% total)
X_ans_train, X_ans_val, X_cust_train, X_cust_val, y_train, y_val = train_test_split(
    X_ans_tmp, X_cust_tmp, y_tmp,
    test_size=0.20, random_state=RANDOM_STATE, stratify=y_tmp
)

print(f'Train : {len(y_train):>6}  ({len(y_train)/len(y)*100:.1f}%)')
print(f'Val   : {len(y_val):>6}  ({len(y_val)/len(y)*100:.1f}%)')
print(f'Test  : {len(y_test):>6}  ({len(y_test)/len(y)*100:.1f}%)')

---
## Step 4 — Fit TF-IDF vectoriser and scale custom features

Both are fit **on training data only**.

In [ ]:
# --- TF-IDF ---
tfidf = TfidfVectorizer(
    max_features=5_000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.85,       # drop near-universal HC3 signature phrases
    sublinear_tf=True,
)
X_train_tfidf = tfidf.fit_transform(X_ans_train)
X_val_tfidf   = tfidf.transform(X_ans_val)
X_test_tfidf  = tfidf.transform(X_ans_test)

print(f'TF-IDF vocab size: {len(tfidf.vocabulary_)}')

# --- Custom features ---
def prep_custom(X):
    return X.replace([np.inf, -np.inf], 0).fillna(0)

X_cust_train_clean = prep_custom(X_cust_train)
X_cust_val_clean   = prep_custom(X_cust_val)
X_cust_test_clean  = prep_custom(X_cust_test)

scaler = StandardScaler(with_mean=False)
X_train_custom = scaler.fit_transform(X_cust_train_clean)
X_val_custom   = scaler.transform(X_cust_val_clean)
X_test_custom  = scaler.transform(X_cust_test_clean)

feature_cols = X_cust_train_clean.columns.tolist()
print(f'Custom features: {feature_cols}')

---
## Step 5 — Train two separate logistic regression models

Each uses `LogisticRegressionCV` to find its own best `C` independently.
This is important — the right regularisation strength is different for a 5,000-dimensional
sparse TF-IDF matrix vs an 8-dimensional dense stylometric feature vector.

In [ ]:
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Model 1: TF-IDF only
print('Training TF-IDF model...')
model_tfidf = LogisticRegressionCV(
    Cs=10,
    cv=cv_splitter,
    penalty='l2',
    solver='saga',
    max_iter=2000,
    class_weight='balanced',
    scoring='f1_macro',
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
model_tfidf.fit(X_train_tfidf, y_train)
print(f'  Best C (TF-IDF model): {model_tfidf.C_[0]:.6f}')

# Model 2: Custom stylometric features only
print('Training custom features model...')
model_custom = LogisticRegressionCV(
    Cs=10,
    cv=cv_splitter,
    penalty='l2',
    solver='saga',
    max_iter=2000,
    class_weight='balanced',
    scoring='f1_macro',
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
model_custom.fit(X_train_custom, y_train)
print(f'  Best C (custom model):  {model_custom.C_[0]:.6f}')

---
## Step 6 — Individual model performance on validation set

Before combining, we see what each model achieves alone.
This tells you how much each side is contributing.

In [ ]:
pred_tfidf_val  = model_tfidf.predict(X_val_tfidf)
pred_custom_val = model_custom.predict(X_val_custom)

f1_tfidf  = f1_score(y_val, pred_tfidf_val,  average='macro')
f1_custom = f1_score(y_val, pred_custom_val, average='macro')

print(f'Validation F1 — TF-IDF model only:  {f1_tfidf:.4f}')
print(f'Validation F1 — Custom model only:   {f1_custom:.4f}')
print()
print('--- TF-IDF model (val) ---')
print(classification_report(y_val, pred_tfidf_val, target_names=['Human', 'AI']))
print('--- Custom model (val) ---')
print(classification_report(y_val, pred_custom_val, target_names=['Human', 'AI']))

---
## Step 7 — Alpha scan on validation set

We scan `alpha` from 0.0 to 1.0 in steps of 0.05.

```
final_prob = alpha * prob_custom + (1 - alpha) * prob_tfidf
```

The best alpha by F1-macro on the validation set is chosen.
The test set is **not touched** here.

### How to interpret the results:
- If best alpha is near **0.0**: TF-IDF dominates — but watch for generalisation failure
- If best alpha is near **1.0**: custom features dominate — more generalisable but lower benchmark score
- If best alpha is near **0.5**: both contribute equally — balanced

For real-world robustness you may want to **manually choose a higher alpha** than the
validation-optimal one, accepting a small benchmark drop in exchange for better generalisation.

In [ ]:
# Get probability of class 1 (AI) from each model on val set
prob_tfidf_val  = model_tfidf.predict_proba(X_val_tfidf)[:, 1]
prob_custom_val = model_custom.predict_proba(X_val_custom)[:, 1]

alphas     = np.arange(0.0, 1.05, 0.05)
val_scores = []

for alpha in alphas:
    combined_prob = alpha * prob_custom_val + (1 - alpha) * prob_tfidf_val
    preds         = (combined_prob > 0.5).astype(int)
    score         = f1_score(y_val, preds, average='macro')
    val_scores.append(score)

val_scores = np.array(val_scores)
best_idx   = np.argmax(val_scores)
best_alpha = alphas[best_idx]

print('Alpha scan results (validation F1-macro):')
print(f'{"alpha":>8}  {"F1-macro":>10}')
print('-' * 22)
for a, s in zip(alphas, val_scores):
    marker = '  <-- best' if abs(a - best_alpha) < 1e-9 else ''
    print(f'{a:8.2f}  {s:10.4f}{marker}')

print()
print(f'Best alpha on validation: {best_alpha:.2f}')
print()
print('Interpretation:')
if best_alpha < 0.2:
    print('  TF-IDF is dominating. Consider manually raising alpha (e.g. 0.4) for real-world robustness.')
elif best_alpha > 0.8:
    print('  Custom features are dominating. TF-IDF is adding little value on this data.')
else:
    print('  Both models are contributing. Alpha is well-balanced.')

---
## Step 8 — Choose your final alpha

You have two options:

- **Use `best_alpha`** — maximises benchmark F1 on validation
- **Use `robust_alpha`** — manually set higher to favour custom features for real-world generalisation

Change `USE_ROBUST_ALPHA = True` and set `robust_alpha` to your preferred value.

In [ ]:
USE_ROBUST_ALPHA = False   # set True to override with a more generalisable alpha
robust_alpha     = 0.6     # adjust: higher = more weight on custom features

final_alpha = robust_alpha if USE_ROBUST_ALPHA else best_alpha
print(f'Final alpha selected: {final_alpha:.2f}')
print(f'  → TF-IDF weight:  {1 - final_alpha:.2f}')
print(f'  → Custom weight:  {final_alpha:.2f}')

---
## Step 9 — Final evaluation on held-out test set

This is the only time the test set is used.

In [ ]:
prob_tfidf_test  = model_tfidf.predict_proba(X_test_tfidf)[:, 1]
prob_custom_test = model_custom.predict_proba(X_test_custom)[:, 1]

combined_prob_test = final_alpha * prob_custom_test + (1 - final_alpha) * prob_tfidf_test
y_pred_combined    = (combined_prob_test > 0.5).astype(int)

# Also evaluate each model alone on test for comparison
y_pred_tfidf_only  = model_tfidf.predict(X_test_tfidf)
y_pred_custom_only = model_custom.predict(X_test_custom)

print('=' * 55)
print(f'  Test results  (alpha = {final_alpha:.2f})')
print('=' * 55)

print(f'\n--- Combined model (alpha={final_alpha:.2f}) ---')
print(classification_report(y_test, y_pred_combined, target_names=['Human', 'AI']))
print(f'Accuracy: {accuracy_score(y_test, y_pred_combined):.4f}')
print(f'F1-macro: {f1_score(y_test, y_pred_combined, average="macro"):.4f}')

print(f'\n--- TF-IDF only (alpha=0.00) ---')
print(f'Accuracy: {accuracy_score(y_test, y_pred_tfidf_only):.4f}')
print(f'F1-macro: {f1_score(y_test, y_pred_tfidf_only, average="macro"):.4f}')

print(f'\n--- Custom only (alpha=1.00) ---')
print(f'Accuracy: {accuracy_score(y_test, y_pred_custom_only):.4f}')
print(f'F1-macro: {f1_score(y_test, y_pred_custom_only, average="macro"):.4f}')

print(f'\nConfusion matrix (combined):')
print(confusion_matrix(y_test, y_pred_combined))

---
## Step 10 — Alpha sensitivity table

Shows test F1 at every alpha value so you can make an informed trade-off decision.
The benchmark drop from moving alpha up (more custom features) tells you the cost
of choosing robustness over memorisation.

In [ ]:
print(f'{"alpha":>8}  {"Test F1-macro":>14}  {"Test Accuracy":>14}  note')
print('-' * 60)
for alpha in alphas:
    cp     = alpha * prob_custom_test + (1 - alpha) * prob_tfidf_test
    preds  = (cp > 0.5).astype(int)
    f1     = f1_score(y_test, preds, average='macro')
    acc    = accuracy_score(y_test, preds)
    note   = ''
    if abs(alpha - best_alpha) < 1e-9:
        note = '<-- val-optimal'
    elif abs(alpha - final_alpha) < 1e-9 and USE_ROBUST_ALPHA:
        note = '<-- robust choice'
    print(f'{alpha:8.2f}  {f1:14.4f}  {acc:14.4f}  {note}')

---
## Step 11 — Save artifacts

In [ ]:
joblib.dump(model_tfidf,  'model_tfidf.pkl')
joblib.dump(model_custom, 'model_custom.pkl')
joblib.dump(tfidf,        'tfidf_vectorizer.pkl')
joblib.dump(scaler,       'feature_scaler.pkl')
joblib.dump(feature_cols, 'feature_columns.pkl')
joblib.dump(final_alpha,  'final_alpha.pkl')

print('Saved artifacts:')
for f in ['model_tfidf.pkl','model_custom.pkl','tfidf_vectorizer.pkl',
          'feature_scaler.pkl','feature_columns.pkl','final_alpha.pkl']:
    print(f'  {f}')

---
## Step 12 — Inference

Mirrors training exactly: full-text feature extraction, no chunking.
Uses the saved `final_alpha` to combine probabilities.

In [ ]:
def predict_text(text, model_tfidf, model_custom, tfidf, scaler, feature_cols, alpha):
    """
    Predict AI (1) vs Human (0) for a piece of text.

    alpha controls the weight on the custom (stylometric) model:
        final_prob = alpha * prob_custom + (1 - alpha) * prob_tfidf

    Higher alpha = more weight on generalisable stylometric features.
    Lower alpha  = more weight on vocabulary/phrase patterns.
    """
    text_clean = clean_text(text.lower())

    # Custom features
    sentences  = split_sentences(text_clean)
    feats      = aggregate_sentence_features(sentences)
    feat_df    = pd.DataFrame([feats]).replace([np.inf, -np.inf], 0).fillna(0)
    feat_df    = feat_df.reindex(columns=feature_cols, fill_value=0)
    custom_scaled = scaler.transform(feat_df)

    # TF-IDF features
    text_tfidf = tfidf.transform([text_clean])

    # Probabilities from each model
    p_tfidf  = model_tfidf.predict_proba(text_tfidf)[0, 1]
    p_custom = model_custom.predict_proba(custom_scaled)[0, 1]

    # Weighted combination
    p_final    = alpha * p_custom + (1 - alpha) * p_tfidf
    prediction = 1 if p_final > 0.5 else 0

    return {
        'prediction'   : 'AI' if prediction == 1 else 'Human',
        'prob_ai'      : round(float(p_final), 4),
        'prob_human'   : round(1 - float(p_final), 4),
        'prob_ai_tfidf'  : round(float(p_tfidf), 4),
        'prob_ai_custom' : round(float(p_custom), 4),
        'alpha_used'   : alpha,
    }


# --- Demo ---
sample_ai = """In this pipeline, I am performing data cleaning before passing features
into the model. First, I replace all missing values with 0. Then I handle infinite values
by replacing them with 0, since such values can occur due to division operations in
feature engineering. Finally, I align the feature DataFrame with the exact column order
used during training by reindexing using the saved feature list."""

sample_human = """honestly i just googled it lol. the way i think about it is like,
imagine you're trying to split a pizza — if you can't do it evenly then you need to
figure out what the common factor is. my teacher explained it way better but basically
that's the gist of it. might be wrong tho haha"""

print('=== Inference Demo ===')
print()
r1 = predict_text(sample_ai,    model_tfidf, model_custom, tfidf, scaler, feature_cols, final_alpha)
r2 = predict_text(sample_human, model_tfidf, model_custom, tfidf, scaler, feature_cols, final_alpha)

print(f'Expected AI    → {r1}')
print(f'Expected Human → {r2}')
print()
print('Note: prob_ai_tfidf and prob_ai_custom show each model\'s individual confidence.')
print('If they disagree strongly, the text is ambiguous — treat the prediction with caution.')